# Import libraries and Link Data 

In [16]:
import os
import glob
import arcpy
from arcpy.sa import *

from config import PROJECT_FOLDER, NLCD_TCC_YEAR

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

raw_folder = os.path.join(PROJECT_FOLDER, "data", "raw")
processed_folder = os.path.join(PROJECT_FOLDER, "data", "processed")
gdb_path = os.path.join(PROJECT_FOLDER, "UrbanHeatEquity.gdb")

landsat_folder = os.path.join(raw_folder, "landsat")

st_paths = sorted(glob.glob(os.path.join(landsat_folder, "*_ST_B10.tif")))
qa_paths = sorted(glob.glob(os.path.join(landsat_folder, "*_QA_PIXEL.tif")))

if len(st_paths) == 0 or len(qa_paths) == 0:
    raise RuntimeError("Missing Landsat ST_B10 or QA_PIXEL rasters.")

if len(st_paths) != len(qa_paths):
    raise RuntimeError("Thermal raster count does not match QA raster count.")

city_boundary = os.path.join(gdb_path, "city_of_phoenix_boundary")
tracts_study = os.path.join(gdb_path, "phoenix_city_intersecting_tracts")

tree_canopy_path = os.path.join(
    raw_folder,
    "nlcd_tree_canopy",
    f"nlcd_tcc_{NLCD_TCC_YEAR}_phoenix.tif"
)

if not os.path.exists(tree_canopy_path):
    raise FileNotFoundError(tree_canopy_path)

print("Thermal rasters:", len(st_paths))
print("QA rasters:", len(qa_paths))
print("Tree canopy:", tree_canopy_path)

Thermal rasters: 2
QA rasters: 2
Tree canopy: C:\Users\ava_gotthard\Documents\UrbanHeatEquity\data\raw\nlcd_tree_canopy\nlcd_tcc_2023_phoenix.tif


# Project tracts and study area to Landsat CRS

In [17]:
landsat_sr = arcpy.Describe(st_paths[0]).spatialReference

tracts_landsat = os.path.join(gdb_path, "phoenix_tracts_landsat_crs")
city_boundary_landsat = os.path.join(gdb_path, "city_boundary_landsat_crs")

for fc in [tracts_landsat, city_boundary_landsat]:
    if arcpy.Exists(fc):
        arcpy.management.Delete(fc)

arcpy.management.Project(tracts_study, tracts_landsat, landsat_sr)
arcpy.management.Project(city_boundary, city_boundary_landsat, landsat_sr)

print("Projected tracts and city boundary.")

Projected tracts and city boundary.


# Convert Landsat scenes to Celsius and apply QA cloud mask

In [18]:
clean_lst_paths = []

for st_path, qa_path in zip(st_paths, qa_paths):

    scene_name = os.path.basename(st_path).replace("_ST_B10.tif", "")
    print("Processing:", scene_name)

    st_raster = Raster(st_path)

    lst_kelvin = st_raster * 0.00341802 + 149.0
    lst_celsius = lst_kelvin - 273.15

    qa = Raster(qa_path)

    bad_pixels = (
        (BitwiseAnd(qa, 2) > 0) |
        (BitwiseAnd(qa, 4) > 0) |
        (BitwiseAnd(qa, 8) > 0) |
        (BitwiseAnd(qa, 16) > 0) |
        (BitwiseAnd(qa, 32) > 0)
    )

    lst_clean = SetNull(bad_pixels, lst_celsius)

    clean_path = os.path.join(processed_folder, f"{scene_name}_lst_clean.tif")

    if arcpy.Exists(clean_path):
        arcpy.management.Delete(clean_path)

    lst_clean.save(clean_path)
    clean_lst_paths.append(clean_path)

print("Cleaned rasters:", len(clean_lst_paths))

Processing: LC08_L2SP_036037_20230927_02_T1
Processing: LC09_L2SP_037037_20230926_02_T1
Cleaned rasters: 2


# Mosaic LST scenes

In [19]:
mosaic_lst = os.path.join(processed_folder, "lst_celsius_mosaic.tif")

if arcpy.Exists(mosaic_lst):
    arcpy.management.Delete(mosaic_lst)

arcpy.management.MosaicToNewRaster(
    input_rasters=clean_lst_paths,
    output_location=processed_folder,
    raster_dataset_name_with_extension="lst_celsius_mosaic.tif",
    coordinate_system_for_the_raster=landsat_sr,
    pixel_type="32_BIT_FLOAT",
    number_of_bands=1,
    mosaic_method="MEAN"
)

print("Mosaic created:", mosaic_lst)

Mosaic created: C:\Users\ava_gotthard\Documents\UrbanHeatEquity\data\processed\lst_celsius_mosaic.tif


# Clip LST mosaic to City of Phoenix

In [20]:
lst_clip_path = os.path.join(processed_folder, "lst_celsius_city_of_phoenix.tif")

if arcpy.Exists(lst_clip_path):
    arcpy.management.Delete(lst_clip_path)

arcpy.management.Clip(
    in_raster=mosaic_lst,
    rectangle="#",
    out_raster=lst_clip_path,
    in_template_dataset=city_boundary_landsat,
    nodata_value="#",
    clipping_geometry="ClippingGeometry",
    maintain_clipping_extent="NO_MAINTAIN_EXTENT"
)

print("LST clipped:", lst_clip_path)

LST clipped: C:\Users\ava_gotthard\Documents\UrbanHeatEquity\data\processed\lst_celsius_city_of_phoenix.tif


# Copy Phoenix-only tree canopy raster to processed folder

In [21]:
canopy_processed = os.path.join(processed_folder, "tree_canopy_city_of_phoenix.tif")

if arcpy.Exists(canopy_processed):
    arcpy.management.Delete(canopy_processed)

arcpy.management.CopyRaster(tree_canopy_path, canopy_processed)

print("Tree canopy copied:", canopy_processed)

Tree canopy copied: C:\Users\ava_gotthard\Documents\UrbanHeatEquity\data\processed\tree_canopy_city_of_phoenix.tif
